# Face AI Pipeline — Stage 1: YOLOv8 Face Detector Training

**GPU-accelerated training on Kaggle T4**

This notebook trains a YOLOv8n model on your face detection dataset.
At the end, `best.pt` is saved — download it and place it at:
`D:\face-ai-pipeline\models\yolo\best.pt`

---
### Before running:
1. Upload your `face-detection-dataset.zip` as a Kaggle Dataset
2. Add it to this notebook via **+ Add Data**
3. Enable GPU: **Settings → Accelerator → GPU T4 x2**
4. **Run All**

## Step 1 — Check GPU & Install Dependencies

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')
print('PyTorch version:', torch.__version__)

!pip install ultralytics -q
import ultralytics
print('Ultralytics version:', ultralytics.__version__)

## Step 2 — Find & Extract Dataset

In [ ]:
import os, zipfile, glob

# Kaggle datasets are mounted under /kaggle/input/
print('Available input files:')
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        path = os.path.join(root, f)
        size_mb = os.path.getsize(path) / 1e6
        print(f'  {path}  ({size_mb:.1f} MB)')

In [ ]:
# Find the zip file (works regardless of exact dataset name)
zip_files = glob.glob('/kaggle/input/**/*.zip', recursive=True)
print('ZIP files found:', zip_files)

DATASET_DIR = '/kaggle/working/dataset'
os.makedirs(DATASET_DIR, exist_ok=True)

if zip_files:
    zip_path = zip_files[0]
    print(f'Extracting: {zip_path}')
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(DATASET_DIR)
    print('Extraction complete!')
else:
    # Dataset might already be extracted (if added as a Kaggle dataset)
    DATASET_DIR = '/kaggle/input'
    print('No ZIP found — using /kaggle/input directly')

# List what we have
print('\nDataset structure:')
for item in os.listdir(DATASET_DIR):
    print(' ', item)

## Step 3 — Auto-detect Dataset Paths & Verify

In [ ]:
# Auto-find images/train folder
train_candidates = glob.glob(f'{DATASET_DIR}/**/images/train', recursive=True)
val_candidates   = glob.glob(f'{DATASET_DIR}/**/images/val',   recursive=True)

if not train_candidates:
    # Try without 'images' subfolder
    train_candidates = glob.glob(f'{DATASET_DIR}/**/train', recursive=True)
    val_candidates   = glob.glob(f'{DATASET_DIR}/**/val',   recursive=True)

TRAIN_IMG_DIR = train_candidates[0] if train_candidates else None
VAL_IMG_DIR   = val_candidates[0]   if val_candidates   else None

print(f'Train images: {TRAIN_IMG_DIR}')
print(f'Val   images: {VAL_IMG_DIR}')

if TRAIN_IMG_DIR:
    train_count = len(os.listdir(TRAIN_IMG_DIR))
    val_count   = len(os.listdir(VAL_IMG_DIR)) if VAL_IMG_DIR else 0
    print(f'Train: {train_count:,} images')
    print(f'Val  : {val_count:,} images')

# The root of the dataset (parent of images/ and labels/)
DATASET_ROOT = os.path.dirname(os.path.dirname(TRAIN_IMG_DIR))
print(f'Dataset root: {DATASET_ROOT}')

## Step 4 — Create YOLO Dataset Config

In [ ]:
import yaml

dataset_cfg = {
    'path':  DATASET_ROOT,
    'train': 'images/train',
    'val':   'images/val',
    'nc':    1,
    'names': {0: 'face'}
}

YAML_PATH = '/kaggle/working/face_dataset.yaml'
with open(YAML_PATH, 'w') as f:
    yaml.dump(dataset_cfg, f, default_flow_style=False)

print('Dataset YAML written to:', YAML_PATH)
print('\nContents:')
with open(YAML_PATH) as f:
    print(f.read())

## Step 5 — Train YOLOv8n on GPU 🚀

In [ ]:
from ultralytics import YOLO

# ── Hyperparameters (GPU-optimised) ──
MODEL      = 'yolov8n.pt'   # nano — fast & accurate for faces
EPOCHS     = 50             # full training run
BATCH      = 32             # T4 can handle 32 comfortably
IMG_SIZE   = 640            # full resolution
PATIENCE   = 10             # early stopping
PROJECT    = '/kaggle/working/runs'
RUN_NAME   = 'face_detector_gpu'

model = YOLO(MODEL)

results = model.train(
    data      = YAML_PATH,
    epochs    = EPOCHS,
    batch     = BATCH,
    imgsz     = IMG_SIZE,
    device    = 0,           # first GPU
    patience  = PATIENCE,
    project   = PROJECT,
    name      = RUN_NAME,
    exist_ok  = True,
    save      = True,
    plots     = True,
    verbose   = True,
)

print('\nTraining complete!')

## Step 6 — Evaluate on Validation Set

In [ ]:
import os

best_weights = f'{PROJECT}/{RUN_NAME}/weights/best.pt'
print(f'Best weights: {best_weights}')
print(f'Exists: {os.path.exists(best_weights)}')
print(f'Size: {os.path.getsize(best_weights)/1e6:.1f} MB')

# Load best model and run validation
best_model = YOLO(best_weights)
metrics = best_model.val(data=YAML_PATH, imgsz=IMG_SIZE, device=0)

print('\n=== VALIDATION METRICS ===')
print(f'mAP50     : {metrics.box.map50:.4f}')
print(f'mAP50-95  : {metrics.box.map:.4f}')
print(f'Precision : {metrics.box.mp:.4f}')
print(f'Recall    : {metrics.box.mr:.4f}')

## Step 7 — Test on a Sample Image

In [ ]:
import glob, random, cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO

best_model = YOLO(best_weights)

# Pick a random val image
val_images = glob.glob(f'{VAL_IMG_DIR}/*.jpg') + glob.glob(f'{VAL_IMG_DIR}/*.png')
test_img   = random.choice(val_images)
print(f'Testing on: {test_img}')

results = best_model.predict(test_img, conf=0.4, imgsz=640, device=0, verbose=False)
result  = results[0]

# Plot
annotated = result.plot()
annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(10, 7))
plt.imshow(annotated_rgb)
plt.axis('off')
plt.title(f'Detected {len(result.boxes)} face(s)', fontsize=14)
plt.tight_layout()
plt.show()

print(f'Faces detected: {len(result.boxes)}')
for i, box in enumerate(result.boxes):
    print(f'  Face {i+1}: conf={box.conf.item():.3f}')

## Step 8 — Package & Download Results

In [ ]:
import shutil, zipfile, os

OUTPUT_ZIP = '/kaggle/working/face_detector_results.zip'
RUN_DIR    = f'{PROJECT}/{RUN_NAME}'

with zipfile.ZipFile(OUTPUT_ZIP, 'w', zipfile.ZIP_DEFLATED) as zf:
    # Add best.pt and last.pt
    for wf in ['best.pt', 'last.pt']:
        wpath = os.path.join(RUN_DIR, 'weights', wf)
        if os.path.exists(wpath):
            zf.write(wpath, f'weights/{wf}')
            print(f'Added: weights/{wf}  ({os.path.getsize(wpath)/1e6:.1f} MB)')

    # Add training plots
    for plot in ['results.png', 'confusion_matrix.png', 'PR_curve.png', 'F1_curve.png']:
        ppath = os.path.join(RUN_DIR, plot)
        if os.path.exists(ppath):
            zf.write(ppath, f'plots/{plot}')
            print(f'Added: plots/{plot}')

    # Add the YAML used for training
    zf.write(YAML_PATH, 'face_dataset.yaml')

print(f'\nOutput ZIP: {OUTPUT_ZIP}')
print(f'ZIP size  : {os.path.getsize(OUTPUT_ZIP)/1e6:.1f} MB')
print('\n=== DOWNLOAD INSTRUCTIONS ===')
print('1. Go to the OUTPUT section (right panel) in Kaggle')
print('2. Download: face_detector_results.zip')
print('3. Extract best.pt -> D:\\face-ai-pipeline\\models\\yolo\\best.pt')

## Done! 🎉

After downloading `face_detector_results.zip`:

```
1. Extract the ZIP
2. Copy  weights/best.pt  ->  D:\face-ai-pipeline\models\yolo\best.pt
3. Run the full pipeline locally:

   D:\face-ai-pipeline\venv\Scripts\python.exe pipeline\run_pipeline.py --image your_image.jpg
```

Your pipeline will now use the GPU-trained YOLOv8 detector automatically!